# Step 3: Model Evaluation
Evaluate the trained model on test data and report metrics.

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import joblib
import boto3
import os
import json
from io import BytesIO

S3_ENDPOINT = os.environ.get("S3_ENDPOINT", "http://minio:9000")
S3_BUCKET = os.environ.get("S3_BUCKET", "pipelines")
AWS_ACCESS_KEY_ID = os.environ.get("AWS_ACCESS_KEY_ID", "minio")
AWS_SECRET_ACCESS_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY", "minio123")

s3 = boto3.client("s3", endpoint_url=S3_ENDPOINT,
                   aws_access_key_id=AWS_ACCESS_KEY_ID,
                   aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
                   verify=False)

MODEL_KEY = os.environ.get("MODEL_KEY", "pipeline-artifacts/model.joblib")
TEST_DATA_KEY = os.environ.get("TEST_DATA_KEY", "pipeline-artifacts/test-data.csv")
METRICS_KEY = os.environ.get("METRICS_KEY", "pipeline-artifacts/metrics.json")

model_obj = s3.get_object(Bucket=S3_BUCKET, Key=MODEL_KEY)
clf = joblib.load(BytesIO(model_obj["Body"].read()))
test_obj = s3.get_object(Bucket=S3_BUCKET, Key=TEST_DATA_KEY)
test_df = pd.read_csv(test_obj["Body"])

target = "approved"
X_test = test_df.drop(columns=[target])
y_test = test_df[target]
print(f"Loaded model and {len(test_df)} test rows from MinIO")

In [ ]:
y_pred = clf.predict(X_test)

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1_score": f1_score(y_test, y_pred, zero_division=0),
}

print("=== Model Evaluation Results ===")
for k, v in metrics.items():
    print(f"{k:>12}: {v:.4f}")

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=["Rejected", "Approved"]))

In [ ]:
s3.put_object(Bucket=S3_BUCKET, Key=METRICS_KEY,
              Body=json.dumps(metrics, indent=2))
print(f"Metrics saved to s3://{S3_BUCKET}/{METRICS_KEY}")